# Capable RAG on a runtime you control — Deep Learning Indaba 2026
### "From Colab to Cluster"

This notebook runs the same code two ways: on hosted Colab, and on a runtime
you control. The wall we are showing is not staged. It is what actually
happened while preparing this talk — a GPU usage-limit lockout and a
mid-run disconnect, both on a completely ordinary Colab session, with no
trick involved. Screenshots later in this notebook are real, not
illustrative.

The fix is not a smaller model and not an API. The fix is the runtime
dropdown, top right: **Connect to a local runtime**, backed by your own GPU
box. The notebook code does not change. `localhost` just becomes your
machine.

Once you cross that line:

- **Sovereignty:** retrieved context and questions stay on your box.
- **Cost:** generation runs on your GPU. No per-token bill, no quota.
- **Control:** nobody's usage limits but your own decide whether the demo runs.

Two models, two jobs: the sentence-transformers embedder runs at **index time**;
`qwen3:30b` runs at **query time only**.


## 1. Install the retrieval stack
`torch` ships with Colab. We add the rest.

In [ ]:
!pip install -q sentence-transformers chromadb arxiv requests

## 2. Which GPU is this runtime?
Run this on the free T4 now, and again after you switch to your local runtime.
The device name and VRAM are the whole story.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
free, total = torch.cuda.mem_get_info()
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")

## 3. Crawl arXiv abstracts (abstracts only, no PDF parsing)

In [ ]:
import arxiv

QUERY = "retrieval augmented generation"
MAX_PAPERS = 50

client = arxiv.Client()
search = arxiv.Search(query=QUERY, max_results=MAX_PAPERS,
                      sort_by=arxiv.SortCriterion.SubmittedDate)

papers = []
for r in client.results(search):
    papers.append({
        "id": r.entry_id.split("/")[-1],
        "title": r.title.strip().replace("\n", " "),
        "abstract": r.summary.strip().replace("\n", " "),
    })

print(f"Pulled {len(papers)} abstracts for: {QUERY!r}")
print("Example:", papers[0]["title"])

## 4. Chunk

In [ ]:
def chunk_text(text, max_words=120, overlap=20):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    out, start = [], 0
    while start < len(words):
        out.append(" ".join(words[start:start + max_words]))
        start += max_words - overlap
    return out

chunks = []
for p in papers:
    for i, c in enumerate(chunk_text(p["abstract"])):
        chunks.append({"chunk_id": f"{p['id']}::{i}", "text": c,
                       "title": p["title"], "paper_id": p["id"]})

print(f"{len(papers)} abstracts -> {len(chunks)} chunks")

## 5. Embed on the GPU, store in a persistent Chroma index
`device="cuda"` puts the embedder on the GPU, so this step uses the hardware too,
not just generation. Indexing is the embedder's only job.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")   # 384-dim, on GPU

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64).tolist()

db = chromadb.PersistentClient(path="./chroma_indaba")
collection = db.get_or_create_collection("arxiv_abstracts")
collection.upsert(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings,
    documents=texts,
    metadatas=[{"title": c["title"], "paper_id": c["paper_id"]} for c in chunks],
)
print(f"Indexed {collection.count()} chunks -> ./chroma_indaba (on disk)")

## 6. Retrieve

In [ ]:
def retrieve(question, k=4):
    q_emb = embedder.encode([question]).tolist()
    hits = collection.query(query_embeddings=q_emb, n_results=k)
    return list(zip(hits["documents"][0], hits["metadatas"][0]))

def build_messages(question, context):
    joined = "\n\n".join(f"[{i}] {doc}" for i, (doc, _) in enumerate(context, 1))
    return [
        {"role": "system", "content": "Answer using only the provided context. Cite sources as [n]. If the context is insufficient, say so."},
        {"role": "user", "content": f"Context:\n{joined}\n\nQuestion: {question}"},
    ]

QUESTION = "What techniques improve retrieval quality in RAG systems?"
context = retrieve(QUESTION)
for i, (_, meta) in enumerate(context, 1):
    print(f"[{i}] {meta['title'][:70]}...")

## 6b. Does it actually work? (stop trusting the vibes)
Retrieval above *ran*. That's not the same as *good*. Right now the only signal you have is "the top result looked plausible" — that's the experimenter's stopping point. An architect asks for a number.

Add 3-4 `(question, relevant_paper_id)` pairs below — 2 are pre-filled, add 1-2 more live using questions you've already tried above. Then measure Precision@k, Recall@k and MRR against them.

In [ ]:
EVAL_SET = [
    {"query": "retrieval augmented generation for open domain QA",
     "relevant_paper_ids": [papers[0]["id"]]},
    {"query": "hallucination in large language models",
     "relevant_paper_ids": [p["id"] for p in papers if "halluc" in p["abstract"].lower()][:1]},
    # add 1-2 more here, live, using a question you already ran above:
    # {"query": "...", "relevant_paper_ids": ["<paper id from a hit you saw above>"]},
]
EVAL_SET = [e for e in EVAL_SET if e["relevant_paper_ids"]]  # drop empty ones

def retrieval_metrics(eval_set, retrieve_fn, k=4):
    precisions, recalls, rr = [], [], []
    for ex in eval_set:
        hits = retrieve_fn(ex["query"], k=k)
        hit_ids = [meta["paper_id"] for _, meta in hits]
        relevant = set(ex["relevant_paper_ids"])
        n_found = len(set(hit_ids) & relevant)
        precisions.append(n_found / k)
        recalls.append(n_found / len(relevant))
        rank = next((i + 1 for i, pid in enumerate(hit_ids) if pid in relevant), None)
        rr.append(1 / rank if rank else 0.0)
    return {
        f"Precision@{k}": sum(precisions) / len(precisions),
        f"Recall@{k}": sum(recalls) / len(recalls),
        "MRR": sum(rr) / len(rr),
    }

print("Dense-only retrieval:", retrieval_metrics(EVAL_SET, retrieve))


**Checkpoint:** would you have caught this by reading the output alone? What's the smallest eval set that's still useful?

## 6c. Fix the gap: add lexical search
Dense embeddings compress exact terminology into fuzzy meaning — a model name, an acronym, a specific method can rank low even when it's an exact string match sitting right there. BM25 catches exact terms; RRF fuses the two rankings without needing to tune a blend weight.

In [ ]:
from rank_bm25 import BM25Okapi

tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)

def retrieve_hybrid(question, k=4, pool=20):
    q_emb = embedder.encode([question]).tolist()
    dense_hits = collection.query(query_embeddings=q_emb, n_results=pool)
    dense_ids = dense_hits["ids"][0]

    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_ranked = sorted(range(len(chunks)), key=lambda i: -bm25_scores[i])[:pool]
    bm25_ids = [chunks[i]["chunk_id"] for i in bm25_ranked]

    id_to_chunk = {c["chunk_id"]: c for c in chunks}

    scores = {}
    for rank, cid in enumerate(dense_ids):
        if cid not in id_to_chunk:
            continue   # stale id from an earlier persisted run — not in this session's corpus
        scores[cid] = scores.get(cid, 0) + 1 / (60 + rank)
    for rank, cid in enumerate(bm25_ids):
        scores[cid] = scores.get(cid, 0) + 1 / (60 + rank)

    fused_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return [(id_to_chunk[cid]["text"], id_to_chunk[cid]) for cid in fused_ids]
print("Hybrid (dense + BM25 + RRF):", retrieval_metrics(EVAL_SET, retrieve_hybrid))


**Checkpoint:** what kinds of queries would BM25 actively hurt instead of help?

## 7. Serve the SLM with Ollama
`qwen3:30b` is served locally through Ollama. On a well-resourced local GPU
this generates in well under two minutes per answer. Whether that number
holds up on a Colab-hosted T4 is genuinely untested as of this writing —
and that is not really the point. The point is what happens next.

A single generation on `qwen3:30b` can run long enough to sit inside the
window where two separate Colab limits can end the session for reasons that
have nothing to do with the model or the GPU's capability:

- **GPU usage limits**: Colab's free tier can refuse to connect you to a GPU
  at all, for a cooldown period entirely outside your control ("You cannot
  currently connect to a GPU due to usage limits").
- **Runtime disconnect**: a session can be dropped mid-run for inactivity or
  hitting a maximum duration, independent of whether the job itself was
  still healthy.

Neither of these is a hardware ceiling you can architect around with a
smaller model or a quantised build. They are policy limits on infrastructure
you do not own. That is the actual argument for a runtime you control — not
that Colab is slow, but that Colab can simply stop, on a schedule nobody in
the room can see coming.

First run on a fresh runtime downloads the model, so pre-pull before you
present regardless of which runtime you end up demoing on.


In [ ]:
import os
import shutil
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"


def has(cmd):
    return shutil.which(cmd) is not None


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


def ensure_ollama_installed():
    if has("ollama"):
        print("✓ Ollama installed")
        return

    print("Installing Ollama...")
    run("curl -fsSL https://ollama.com/install.sh | sh")

    if not has("ollama"):
        raise RuntimeError("Ollama installation failed.")


def ollama_ready():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/version", timeout=1)
        return r.status_code == 200
    except Exception:
        return False


def ensure_ollama_running():
    if ollama_ready():
        print("✓ Ollama already running")
        return

    print("Starting Ollama...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=os.environ.copy(),
    )

    for _ in range(60):
        if ollama_ready():
            print("✓ Ollama ready")
            return
        time.sleep(1)

    raise RuntimeError("Timed out waiting for Ollama.")


def ensure_model(model):
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    if model not in result.stdout:
        print(f"Pulling {model}...")
        subprocess.run(["ollama", "pull", model], check=True)
    else:
        print(f"✓ {model} already installed")


# ----------------------------
# Main
# ----------------------------

required = {
    "zstd": "zstd",
    "lspci": "pciutils",
    "lshw": "lshw",
}

missing = [pkg for cmd, pkg in required.items() if not has(cmd)]

if missing:
    run("apt-get update")
    run(f"apt-get install -y {' '.join(sorted(set(missing)))}")

ensure_ollama_installed()
ensure_ollama_running()

MODEL = "qwen3:30b"
ensure_model(MODEL)

print("Ready.")

In [ ]:
import requests, time

def answer_local(question, context, model=MODEL):
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": model, "messages": build_messages(question, context),
                            "stream": False}, timeout=600)
    r.raise_for_status()
    return r.json()

t0 = time.time()
resp = answer_local(QUESTION, context)
dt = time.time() - t0
tok = resp.get("eval_count", 0)

print(resp["message"]["content"])
print(f"\n{tok} tokens in {dt:.1f}s  ->  {tok/max(dt,1e-9):.1f} tok/s")
print("\nWhere is the model actually running:")
!ollama ps

## 7b. You claimed persistence. Prove it.
`chromadb.PersistentClient` supposedly survives a runtime reset. That's an assertion, not a demonstration — and this workshop doesn't take assertions. Run the cell below to confirm you get a hit. Then:
**`Runtime > Disconnect and delete runtime`**, reconnect, and re-run *only* the query cell below — do **not** re-run the embed cell.

If it still answers, the bytes on disk survived even though every Python variable in this session just got wiped. That's the second half of the Colab Ceiling: it's not only compute that's ephemeral, session state is too, unless you deliberately put data somewhere that outlives the runtime.

In [ ]:
# Run this BEFORE disconnecting, and AGAIN after reconnecting.
# After reconnecting, re-run the imports + db/collection-open cells (they
# just open the existing ./chroma_indaba directory) but NOT the embed/
# upsert cell -- that is the whole point.

check = collection.query(query_texts=["retrieval augmented generation"], n_results=1)
print("Chunks in collection:", collection.count())
print("Top hit survived the reset:", check["documents"][0][0][:120], "...")


**Checkpoint:** what exactly survived the disconnect, and why? (Hint: `embedder`, `chunks`, `papers` did not.)

## 8. The fix: connect Colab to your local runtime
No code changes. On your GPU box:

```
pip install jupyter_http_over_ws
jupyter server extension enable --py jupyter_http_over_ws
jupyter notebook \
  --NotebookApp.allow_origin='https://colab.research.google.com' \
  --port=8888 --NotebookApp.port_retries=0
```

Copy the printed `http://localhost:8888/?token=...` URL. In Colab, use the
runtime dropdown (top right, the one in your screenshot) → **Connect to a local
runtime** → paste the URL. Re-run cells 2 and 7.

Cell 2 now reports your 4090 and 24 GB. `ollama ps` shows 100% GPU, the fp16 7B
fits, tokens fly, and nothing left your box.

## 9. Contrast: swap generation for a paid API
Everything up to retrieval is identical. Swap the local call for a hosted one
and the same context leaves your infrastructure and the meter starts. Inert
without a key.

> Needs `pip install openai` and `OPENAI_API_KEY`.

In [ ]:
import os

def answer_hosted(question, context, model="gpt-4o-mini"):
    from openai import OpenAI      # the moment your data leaves the box
    client = OpenAI()
    resp = client.chat.completions.create(model=model, messages=build_messages(question, context))
    return resp.choices[0].message.content

if os.getenv("OPENAI_API_KEY"):
    print(answer_hosted(QUESTION, context))
else:
    print("No OPENAI_API_KEY set, skipping. The local path never needed one.")

## 9b. Is the answer actually grounded?
Fluent and confident is not the same as correct. This judges the answer you already generated above against the evidence you already retrieved — using the **same local Ollama model**, so no new API key and no new network dependency. If the earlier answer came from the model while it was still spilling to CPU, this is a good place to catch something a quick read wouldn't.

In [ ]:
import json as _json

JUDGE_PROMPT = """You are grading an AI-generated answer against the evidence \
it was given. Respond with ONLY a JSON object, no other text:
{{
  "correctness": <1-5>,
  "groundedness": <1-5, does every claim trace back to the evidence>,
  "completeness": <1-5>,
  "relevance": <1-5>,
  "hallucination_detected": <true/false>,
  "explanation": "<one sentence, name the specific claim if hallucination_detected is true>"
}}

QUESTION: {question}

EVIDENCE:
{evidence}

ANSWER TO GRADE:
{answer}
"""

def judge_answer(question, answer, context, model=MODEL):
    evidence = "\n\n".join(f"[{i}] {doc}" for i, (doc, _) in enumerate(context, 1))
    prompt = JUDGE_PROMPT.format(question=question, evidence=evidence, answer=answer)
    r = requests.post("http://localhost:11434/api/chat",
                       json={"model": model,
                             "messages": [{"role": "user", "content": prompt}],
                             "stream": False}, timeout=600)
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    try:
        return _json.loads(raw[raw.index("{"):raw.rindex("}") + 1])
    except Exception:
        return {"error": "judge did not return valid JSON", "raw": raw}

report = judge_answer(QUESTION, resp["message"]["content"], context)
print(_json.dumps(report, indent=2))


In [ ]:
# FALLBACK if the live judge call is slow on stage -- a pre-baked example
# with a genuine unsupported claim, ready to swap in without breaking pace.

FALLBACK_REPORT = {
    "correctness": 3,
    "groundedness": 2,
    "completeness": 4,
    "relevance": 5,
    "hallucination_detected": True,
    "explanation": "The answer states a specific accuracy percentage that does not appear anywhere in the retrieved evidence."
}
print("Fallback example (use if the live call above is slow):")
print(_json.dumps(FALLBACK_REPORT, indent=2))


**Checkpoint:** if this had failed on stage, what would you have shipped instead?

---
**Standalone checkpoint (no code):** what's your fallback if the Ollama call hangs right now, mid-demo?

## Recap for the room

- **Colab is excellent for experimentation, but it has operational limits.** Runtime timeouts and GPU quotas eventually become constraints.
- **The notebook didn't change; the runtime did.** The same workflow ran against a local GPU runtime without changing the RAG pipeline.
- **Infrastructure matters as much as models.** Where you run your application determines what you can build and operate.
- **Different models have different roles.** The embedding model built the index, while the SLM answered questions using retrieved context.
- **Retrieval should be evaluated, not assumed.** Comparing dense retrieval with BM25 + RRF showed why retrieval quality directly affects answers.
- **Persistence matters.** Keeping indexes and models on disk avoids rebuilding work after a runtime restart.
- **Grounding matters.** A local evaluation model helped verify answers against the retrieved evidence rather than relying on fluency alone.